## Why Can't We Train on Entire Datasets?

- Training on the full dataset can be memory-intensive and computationally slow. Mini-batches reduce memory requirements, allow more frequent parameter updates, and often improve generalization due to the noise introduced in gradient estimates.

### Why not load millions of samples into memory at once?

1. Memory constraints:
   Large datasets may not fit into RAM or GPU memory.

2. Frequent parameter updates:
   Mini-batches allow many updates per epoch instead of a single update.

3. Better generalization:
   Gradient estimates from mini-batches contain useful noise that often improves generalization.

4. Computational efficiency:
   Very large batches increase memory and computation costs while providing diminishing improvements in gradient accuracy.

5. Better hardware utilization:
   GPUs are optimized for processing batches rather than individual samples.

### What problems arise with very large datasets?
- Memory constrains and Computational inefficiency arises.

## What Is a Dataset?

- A Dataset is an object that provides access to training examples. Each training example usually contains (features, label)

### What information should a Dataset provide?
-  Number of samples `__len__` and A way to retrieve a sample `__getitem__`.

### Why separate data storage from training logic?
- Separating data storage from training logic makes the code modular, reusable, and scalable. The training loop only needs batches of data, while the Dataset is responsible for loading and providing individual samples regardless of where the data is stored.

## Custom Dataset

In [19]:
import torch 
from torch.utils.data import Dataset

class SimpleDataset(Dataset):
    
    def __init__(self):
        
        self.X = torch.tensor([[1.0], 
                          [2.0], 
                          [3.0], 
                          [4.0]])
        
        self.y = torch.tensor([[3.0], 
                          [5.0], 
                          [7.0], 
                          [9.0]])
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    
obj = SimpleDataset()
print(obj.__len__())
print(obj[2])

4
(tensor([3.]), tensor([7.]))


### Why must Dataset implement `__len__` and `__getitem__`?
- A Dataset must implement `__len__()` so PyTorch knows how many samples exist, and `__getitem__()` so PyTorch can retrieve a specific sample by index. These methods provide a standard interface that DataLoader can use regardless of how the data is stored.

### What does `idx` represent?
- idx represents the position of a sample within the dataset. When DataLoader requests data, it passes an index to `__getitem__()`, which returns the corresponding sample.

## Access Individual Samples

In [20]:
dataset = SimpleDataset()

print(dataset[1])
print(dataset[2])

(tensor([2.]), tensor([5.]))
(tensor([3.]), tensor([7.]))


### What does each call return?
- Each call returns a single training sample corresponding to the requested index. In supervised learning, this sample is usually returned as (features, label).

### Why is each sample returned as (X, y)?
- Each sample is returned as (X, y) because supervised learning requires both the input features and the expected target. The model uses X to make predictions and y to compute the loss during training.

## What Is a DataLoader?

- A DataLoader is an object that takes a Dataset retrieves samples from it, groups them into batches, optionally shuffles them and finally feeds them to the training loop.

- Dataset = warehouse storing products

- DataLoader = worker bringing boxes to the assembly line in batches

- Training Loop = assembly line

### What problem does DataLoader solve?
- DataLoader automates retrieving samples from a Dataset and provides efficient iteration through the dataset in mini-batches. It also handles shuffling, batching, and other data-loading tasks required during training.

### Why not iterate Dataset manually?
- Iterating over a Dataset manually would require implementing batching, shuffling, and sample management ourselves. DataLoader provides these capabilities automatically and efficiently. It also supports mini-batch training, which reduces memory usage and allows more frequent parameter updates than full-batch training.

## Create DataLoader

In [21]:
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True
)

### What does `batch_size` mean?
- `batch_size` specifies how many samples are processed together before computing a gradient update.

### Why might shuffling be useful?
- Shuffling prevents the model from seeing samples in the same order every epoch. This reduces the risk of learning patterns that depend on data ordering and helps batches become more representative of the overall dataset.

## Inspect Batches

In [22]:
for X_batch, y_batch in loader:
    print(X_batch)
    print(X_batch.shape)
    print(y_batch)
    print(y_batch.shape)
    print()

tensor([[2.],
        [1.]])
torch.Size([2, 1])
tensor([[5.],
        [3.]])
torch.Size([2, 1])

tensor([[4.],
        [3.]])
torch.Size([2, 1])
tensor([[9.],
        [7.]])
torch.Size([2, 1])



### How many batches are created?
- 2 batches are created of size 2.

### What shapes do batch tensors have?
- Shape of batch tensor will be (batch_size, features) or (batch_size, targets).

## Batch Size Intuition

Compare:  
batch_size = 1  
batch_size = 50  
batch_size = 500  

for dataset of 500 samples 

- Case 1: batch_size = 1

    - The gradient is actually computed from the loss of a single sample, which makes it a very noisy estimate of the true dataset gradient.

    - This is Stochastic Gradient Descent For 500 samples you get 500 updates per epoch

    - Characteristics:

        - very noisy gradients
        - many updates
        - low memory usage

- Case 2: batch_size = 500

    - Gradient of loss is calculated using entire dataset and with that gradient parameters are updated. Also this is the most accurate estimate of the dataset gradient.

    - This is Batch Gradient Descent For one epoch 1 update only

    - Characteristics:

        - stable gradients
        - high memory usage
        - few updates

- Case 3: batch_size = 50

    - Gradient of loss is calculated using all the samples in single batch of a dataset and with that gradient parameters are updated.

    - This is Mini-Batch Gradient Descent For 500 samples you get 10 updates per epoch

    - Characteristics:
        - reasonable memory usage
        - reasonably stable gradients
        - reasonably frequent updates

- Modern deep learning typically uses mini-batch gradient descent because it provides a practical balance between memory usage, training speed, and gradient quality.

## Epoch vs Batch

### Define:

- sample : single data point within a dataset, typically represented as a single row in a data matrix.
- batch : A batch is a group of samples processed together in a single forward and backward pass.
- epoch : Epoch refers to one complete pass through the entire training dataset.
- iteration : Iteration refers to one single update of the model’s parameters (weights and biases) after processing a mini-batch of data.

## Training Loop with DataLoader

In [23]:
model = torch.nn.Linear(1, 1)

loss_fn = torch.nn.MSELoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01
)

In [24]:
for epoch in range(100):
    
    for X_batch, y_batch in loader:
        
        pred = model(X_batch)
    
        loss = loss_fn(pred, y_batch)
        
        optimizer.zero_grad()
        
        loss.backward()
        
        optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}: Loss = {loss.item():.4f}")


Epoch 0: Loss = 11.9670
Epoch 10: Loss = 0.0042
Epoch 20: Loss = 0.0002
Epoch 30: Loss = 0.0002
Epoch 40: Loss = 0.0002
Epoch 50: Loss = 0.0003
Epoch 60: Loss = 0.0005
Epoch 70: Loss = 0.0002
Epoch 80: Loss = 0.0004
Epoch 90: Loss = 0.0002


### How is this different from the previous notebook?
- In the previous notebook, the entire dataset was passed to the model in a single forward and backward pass, resulting in Batch Gradient Descent. In this notebook, DataLoader splits the dataset into mini-batches, allowing multiple parameter updates within a single epoch.

### Why does deep learning almost always use mini-batches?
- Deep learning almost always uses mini-batches because they reduce memory usage, make efficient use of hardware such as GPUs, allow more frequent parameter updates than full-batch training, and introduce useful gradient noise that often improves generalization.

## Shuffle Experiment

### What changes?
- If `shuffle=False`, samples are retrieved in the same order every epoch.

- If `shuffle=True`, samples are randomly reordered before batches are created, so the composition and order of batches changes each epoch.

### Why might ordered data create problems?
- If the dataset is ordered, consecutive batches may contain similar samples or even only one class. This can bias gradient updates and make training less stable. Shuffling helps ensure batches are more representative of the overall dataset.

## Data Pipeline Mental Model

Dataset -> DataLoader -> Batch -> Model -> Loss -> Update